<a href="https://colab.research.google.com/github/Jerson09/Informe_de_resultados_de_tratamiento_de_audio_Jerson/blob/main/tratamientoImagen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import cv2, os, numpy as np, pandas as pd

RUTA = "/mnt/user-data/uploads/"
IMAGENES = {
    "1_referencia":     "WhatsApp Image 2026-10-16 at 6.09.17 PM.jpeg",
    "2_movimiento":     "WhatsApp Image 2026-10-16 at 6.09.18 PM (1).jpeg",
    "3_angulo":         "WhatsApp Image 2026-10-16 at 6.09.18 PM (2).jpeg",
    "4_fondo_complejo": "WhatsApp Image 2026-10-16 at 6.09.18 PM (3).jpeg",
    "5_baja_luz":       "WhatsApp Image 2026-10-16 at 6.09.18 PM.jpeg",
}

# Verifica que todas existan antes de continuar
for k, f in IMAGENES.items():
    assert os.path.exists(RUTA + f), f"No se encontró {RUTA + f}"

# Región de interés (ROI) aproximada del objeto: (x1, y1, x2, y2)
ROI = {
    "1_referencia": (180, 350, 540, 940),
    "2_movimiento": (0, 500, 380, 910),
    "3_angulo": (20, 190, 610, 760),
    "4_fondo_complejo": (175, 455, 530, 860),
    "5_baja_luz": (330, 360, 715, 880),
}
def nitidez_roi(img, roi):
    x1, y1, x2, y2 = roi
    g = cv2.cvtColor(img[y1:y2, x1:x2], cv2.COLOR_BGR2GRAY)
    return round(float(cv2.Laplacian(g, cv2.CV_64F).var()), 1)

def metricas(img, ruta):
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gris.shape
    return {
        "resolucion": f"{w}x{h}",
        "canales": img.shape[2],
        "brillo_medio": round(float(gris.mean()), 1),
        "contraste_std": round(float(gris.std()), 1),
        "nitidez_lap_var": round(float(cv2.Laplacian(gris, cv2.CV_64F).var()), 1),
        "pct_pix_oscuros(<50)": round(float((gris < 50).mean()*100), 1),
        "nitidez_ROI": None,
        "formato": os.path.splitext(ruta)[1].upper().strip("."),
        "tam_KB": round(os.path.getsize(ruta)/1024, 1),
    }

filas = {}
imgs = {}
for k, f in IMAGENES.items():
    r = RUTA + f
    imgs[k] = cv2.imread(r)
    filas[k] = metricas(imgs[k], r)
    filas[k]["nitidez_ROI"] = nitidez_roi(imgs[k], ROI[k])
df = pd.DataFrame(filas).T
print(df.to_string())
df.to_csv("tabla.csv")

# ---- Tratamiento: baja iluminación (corrección gamma + CLAHE sobre L)
def tratar(img, gamma=0.45, clip=3.0):
    lut = np.array([((i/255.0)**gamma)*255 for i in range(256)]).astype("uint8")
    g = cv2.LUT(img, lut)
    lab = cv2.cvtColor(g, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

orig = imgs["5_baja_luz"]
mej = tratar(orig)
cv2.imwrite("baja_luz_tratada.png", mej)
cv2.imwrite("baja_luz_original.png", orig)
res = {}
for nombre, im in [("antes", orig), ("despues", mej)]:
    g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    res[nombre] = {
        "brillo_medio": round(float(g.mean()),1),
        "contraste_std": round(float(g.std()),1),
        "nitidez_lap_var": round(float(cv2.Laplacian(g, cv2.CV_64F).var()),1),
        "pct_pix_oscuros(<50)": round(float((g<50).mean()*100),1),
        "nitidez_ROI": nitidez_roi(im, ROI["5_baja_luz"]),
        "ruido_est(std de g - blur)": round(float((g.astype(float)-cv2.GaussianBlur(g,(5,5),0)).std()),2),
    }
print(pd.DataFrame(res).to_string())
pd.DataFrame(res).to_csv("antes_despues.csv")

                 resolucion canales brillo_medio contraste_std nitidez_lap_var pct_pix_oscuros(<50) nitidez_ROI formato tam_KB
1_referencia       739x1600       3        216.7          44.3            10.8                  0.0        40.2    JPEG   61.7
2_movimiento       739x1600       3         95.2          35.3             9.1                 11.2        14.7    JPEG   66.2
3_angulo           739x1600       3        104.6          44.5           418.0                 10.9       505.6    JPEG  184.2
4_fondo_complejo   739x1600       3         94.0          49.6           453.8                 24.5       175.0    JPEG  196.7
5_baja_luz         739x1600       3          7.8           6.5             0.3                100.0         1.5    JPEG   15.0
                             antes  despues
brillo_medio                  7.80    58.20
contraste_std                 6.50    27.80
nitidez_lap_var               0.30    15.90
pct_pix_oscuros(<50)        100.00    36.20
nitidez_ROI       